# M2 — QASPER LoRA Model

Public portfolio edition prepared for GitHub and Databricks. Credentials are read from environment variables; research data and generated artifacts are not committed to Git.


In [ ]:
# Databricks uses Unity Catalog Volumes; no Google Drive mount is required.

In [ ]:
# 只在全新环境需要
!pip -q install "transformers==4.44.0" "peft==0.11.1" "accelerate==0.30.1"

In [ ]:
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

In [ ]:
from pathlib import Path

MODEL_ID  = "mistralai/Mistral-7B-Instruct-v0.3"   # 必须与训练时一致
OUT_DIR   = "/Volumes/main/default/thesis_project/M2_NoContext/M2_Test/QASPER_Lora_1.1"  # 你的 LoRA 目录
EVAL_PATH = "/Volumes/main/default/thesis_project/QASPER/processed_20250805_162328/qasper_test_qa_evidence.parquet"
Path(OUT_DIR).exists(), OUT_DIR

8. 载入用于推理的模型（bf16 基座 + 你的 LoRA 适配器）

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

torch.backends.cuda.matmul.allow_tf32 = True
try: torch.set_float32_matmul_precision("high")
except: pass

tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = "right"

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto", low_cpu_mem_usage=True
)

In [ ]:
import os, json, shutil, re
from peft import PeftModel, LoraConfig

ALLOWED = {
    "r","lora_alpha","lora_dropout","target_modules","fan_in_fan_out","bias",
    "use_dora","use_rslora","init_lora_weights","inference_mode",
    "rank_pattern","alpha_pattern","modules_to_save","layers_to_transform","layers_pattern",
    "task_type","peft_type","auto_mapping","base_model_name_or_path","revision",
    "loftq_config","megablocks"
}

def sanitize_and_load_lora(base_model, adapter_dir):
    cfg_path = os.path.join(adapter_dir, "adapter_config.json")
    assert os.path.exists(cfg_path), f"adapter_config.json 不在 {adapter_dir}"

    with open(cfg_path, "r") as f:
        cfg = json.load(f)

    # 备份
    bak = cfg_path + ".bak2"
    shutil.copy(cfg_path, bak)

    # 1) 若有嵌套块，拍平；若是 dora/corda，置 use_dora=True
    for nested_key in ("lora_config", "dora_config", "corda_config"):
        if nested_key in cfg:
            nested = cfg.pop(nested_key)
            if isinstance(nested, dict):
                for k, v in nested.items():
                    cfg.setdefault(k, v)
            if nested_key in ("dora_config", "corda_config"):
                cfg["use_dora"] = True

    # 2) 删掉所有 *_config 奇怪字段（有些可能不是 dict）
    for k in list(cfg.keys()):
        if re.search(r"_config$", k) and k not in ("loftq_config",):  # 保留 loftq_config（PEFT 支持）
            cfg.pop(k, None)

    # 3) 兜底必要字段
    cfg["peft_type"] = "LORA"
    cfg.setdefault("task_type", "CAUSAL_LM")

    # 4) 仅保留 LoraConfig 可识别字段
    clean = {k: v for k, v in cfg.items() if k in ALLOWED}

    # 5) 构造 LoraConfig；如 target_modules 缺失就给一个常见默认（Mistral/Llama）
    if "target_modules" not in clean or not clean["target_modules"]:
        clean["target_modules"] = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

    # 打印检查
    print("Using cleaned LoraConfig keys:", sorted(clean.keys()))

    peft_conf = LoraConfig(**clean)

    # 用“显式配置”加载，绕过磁盘上的异常字段
    model = PeftModel.from_pretrained(base_model, adapter_dir, config=peft_conf).eval()
    return model

# 调用
model = sanitize_and_load_lora(base, OUT_DIR)
print("✅ LoRA adapter loaded.")


In [ ]:
SYSTEM_PROMPT    = "You are a careful research assistant."
USER_INSTRUCTION = "Answer concisely (≤120 words). If unsure, say you don't know."

def build_prompt(q: str) -> str:
    return (
        "<s>[INST] <<SYS>>\n"
        f"{SYSTEM_PROMPT}\n"
        "<</SYS>>\n"
        f"Question: {q}\n"
        f"Instruction: {USER_INSTRUCTION}\n"
        "[/INST]\n"
    )

In [ ]:
GEN_KW = dict(
    max_new_tokens=192,         # 与 M0/M1 对齐
    do_sample=True,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.05,
    eos_token_id=tok.eos_token_id,
    pad_token_id=tok.pad_token_id,
)

In [ ]:
def build_prompt(q: str) -> str:
    return ("<s>[INST] <<SYS>>\nYou are a careful research assistant.\n<</SYS>>\n"
            f"Question: {q}\nInstruction: Answer concisely (≤120 words). If unsure, say you don't know.\n[/INST]\n")

GEN_KW = dict(max_new_tokens=192, do_sample=True, temperature=0.2, top_p=0.9,
              repetition_penalty=1.05, eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)

@torch.inference_mode()
def gen_one(q: str) -> str:
    batch = tok(build_prompt(q), return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**batch, **GEN_KW)
    new_ids = out[0, batch["input_ids"].shape[1]:]
    return tok.decode(new_ids, skip_special_tokens=True).strip()

9. 批量生成 Validation 集预测并保存（带 paper_id/question_id）

In [ ]:
# —— 只需执行一次 ——
import torch, gc
try:
    del trainer  # 如果之前创建过
except:
    pass
gc.collect(); torch.cuda.empty_cache()

# 让基座不要 offload，整机进 GPU（A100-40GB 足够容纳 7B bf16）
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map=None, low_cpu_mem_usage=True
)
model = sanitize_and_load_lora(base, OUT_DIR)  # 用你刚才的“强力修补器”
model.to("cuda").eval()                        # ✅ 关键：整机搬上 GPU

In [ ]:
# 仅用于推理：左填充
tok.padding_side = "left"

# 显式设置输入截断上限（给 tokenizer 用）
MAX_IN_LEN = min(2048, getattr(model.config, "max_position_embeddings", 2048))
# 可选：也告诉 tokenizer 一个最大长度，消除告警
tok.model_max_length = MAX_IN_LEN

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

df = pd.read_parquet(EVAL_PATH)[["paper_id","question_id","question","gold_answer"]].copy()
df["question"] = df["question"].astype(str).str.strip()
df["gold_answer"] = df["gold_answer"].astype(str).str.strip()

BATCH_SIZE = 16  # 先保守一点；稳定后可提到 24
preds = []

for i in tqdm(range(0, len(df), BATCH_SIZE), desc="Generating validation predictions"):
    qs = df["question"].iloc[i:i+BATCH_SIZE].tolist()
    texts = [build_prompt(q) for q in qs]

    # 关键：左填充 + 显式 max_length + 不重复加特符
    batch = tok(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_IN_LEN,
        add_special_tokens=False,
    )
    batch = {k: v.to("cuda") for k, v in batch.items()}

    with torch.inference_mode():
        out = model.generate(**batch, **GEN_KW)

    # 切掉“整条输入长度”（等于批里统一的序列长度）
    seq_len = batch["input_ids"].shape[1]
    for j in range(out.size(0)):
        gen_ids = out[j, seq_len:]
        preds.append(tok.decode(gen_ids, skip_special_tokens=True).strip())

df_pred = df.copy()
df_pred["answer_M2_1"] = preds

In [ ]:
SAVE_DIR = f"{OUT_DIR}/predictions_1.2"
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
csv_p  = f"{SAVE_DIR}/test_M2_1.2_with_ids.csv"
parq_p = f"{SAVE_DIR}/test_M2_1.2_with_ids.parquet"

df_pred.to_csv(csv_p, index=False, encoding="utf-8-sig")
df_pred.to_parquet(parq_p, index=False)
print("✅ Saved:\n -", csv_p, "\n -", parq_p)


In [ ]:
from pathlib import Path
import pandas as pd

# 按你的保存路径
SAVE_DIR = f"{OUT_DIR}/predictions_1.2"
csv_p  = Path(SAVE_DIR) / "test_M2_1.2_with_ids.csv"
parq_p = Path(SAVE_DIR) / "test_M2-1_1.2_with_ids.parquet"

# 选一个可用文件（优先 parquet）
if parq_p.exists():
    path = parq_p
elif csv_p.exists():
    path = csv_p
else:
    raise FileNotFoundError(f"没找到文件：\n- {parq_p}\n- {csv_p}")

# 读取
if path.suffix == ".parquet":
    df = pd.read_parquet(path)
else:
    try:
        df = pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="utf-8-sig")

# 想看的列（按存在的列自动取）
preferred_cols = [
    "paper_id", "question_id", "question", "gold_answer",
    "answer_M2_1", "answer_M2-1", "prediction", "pred", "answer"
]
cols = [c for c in preferred_cols if c in df.columns]
view = df[cols].head(5) if cols else df.head(5)

print("Showing head(5) from:", path)
print(view.to_string(index=False))
